In [50]:
import os, time, gc
import re, tempfile
from IPython.display import HTML
import random
import sys
module_dir = "./src"
sys.path.append(module_dir)
from api import run_mmseqs2
import matplotlib.pyplot as plt
import string
import numpy as np
import pandas as pd
import pickle
from colabdesign.af.contrib import predict
from multiprocessing import Pool, cpu_count
import functools
import json
np.random.seed(123)
random.seed(123)
from get_MSA import process_jobname as run_get_MSA
from util_SMICE import *
from colabdesign import mk_af_model, clear_mem
from colabdesign.shared.protein import _np_rmsd
import shutil  # Added for file operations
import zipfile
from io import BytesIO
from Bio import PDB
from Bio.PDB import PDBIO

In [3]:
with open('./config/config_SMICE_benchmark.json', 'r') as f:
    config = json.load(f)

base_dir = config["base_dir"]
base_output_dir = '/n/kou_lab/yongkai/SMICE/outputs_demo/'
base_result_dir = '/n/kou_lab/yongkai/SMICE/results_demo/'
pdb_seq_file = config["pdb_seq_file"]
MSA_saved_basedir = config["MSA_saved_basedir"]
cov = 75

### set AF2 model

In [4]:
import subprocess
def ColabFold_batch(input_dir, output_dir, models=[1, 2, 3, 4, 5]):
    # Convert list to space-separated string
    models_str = ' '.join(str(model) for model in models)
    # Build the command
    cmd = f'./bash/demo/colabfold.sh "{input_dir}" "{output_dir}" {models_str}'
    # Use subprocess for better control
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    # Optional: Return or check the result
    return result.returncode, result.stdout, result.stderr

### obtain MSA

In [5]:
jobname = "5jytA"
run_get_MSA(jobname)
MSA_saved_dir = f"{MSA_saved_basedir}{jobname}"
msa = np.load(f"{MSA_saved_dir}/msa/msa.npy")
deletion_matrix = np.load(f"{MSA_saved_dir}/msa/del_mat.npy")
N, L = msa.shape


MAPLRKTAVLKLYVAGNTPNSVRALKTLANILEKEFKGVYALKVIDVLKNPQLAEEDKILATPTLAKVLPPPVRRIIGDLSNREKVLIALRLLAEEIGDYKDDDDK
getting unpaired MSA
parsing msas
gathering info
filtering sequences


- 10:30:20.260 INFO: Input file = /n/kou_lab/yongkai/SS_AF2/MSA_cov75_all/5jytA/msa/1.a3m

- 10:30:20.260 INFO: Output file = /n/kou_lab/yongkai/SS_AF2/MSA_cov75_all/5jytA/msa/1.out.a3m



selecting final sequences
(4096, 106)


### run bayesian sequential sampling

In [15]:
save_subMSA_dir = f"{base_output_dir}{jobname}/bss_res/"
os.makedirs(save_subMSA_dir, exist_ok=True)
lamb_list = [0,1,2,3]
model_list = [1]

In [14]:

if os.path.exists(os.path.join(MSA_saved_dir, "msa/msa.npy")):
    msa = np.load(os.path.join(MSA_saved_dir, "msa/msa.npy"))
    if len(msa)<20:
        print("Warning!! Number of MSA is %d"%len(msa))
    else:
        msa_samples_indices = run_BSS(msa, os.path.join(MSA_saved_dir, "msa.a3m"), jobname, save_subMSA_dir,lamb_list,return_msa=True,n_tries = 1)
else:
    print("MSA of %s not found"%jobname)

In [17]:
query_seq=msa[0]
query_dtx = deletion_matrix[0]
ss_msas=[]
ss_dtxs=[]
for ss_indices in msa_samples_indices:
    ss_msa = msa[ss_indices]
    ss_dtx = deletion_matrix[ss_indices]
    ss_msa = np.concatenate([[query_seq], ss_msa])
    ss_dtx = np.concatenate([[query_dtx], ss_dtx])
    ss_msas.append(ss_msa)
    ss_dtxs.append(ss_dtx)


In [18]:
print("%d MSA subsets are sampled in sequential sampling"%len(msa_samples_indices))

155 MSA subsets are sampled in sequential sampling


In [14]:
for lamb in lamb_list:
    if N<100:
        n_neighbors_list = [10]
    else:
        n_neighbors_list = [10,30]
    for n_neighbors in n_neighbors_list: 
        input_dir=f"{base_output_dir}{jobname}/bss_res/msa_ss_bayes_lamb{lamb}_neighbors{n_neighbors}/"
        output_dir=f"{base_output_dir}{jobname}/bss_res/pdb_ss_bayes_colab_lamb{lamb}_neighbors{n_neighbors}/"
        res_BSS = ColabFold_batch(input_dir, output_dir)

### Enhanced Sampling

In [17]:
aa_order = {'A': 0, 'R': 1, 'N': 2, 'D': 3, 'C': 4, 'Q': 5, 'E': 6, 'G': 7, 'H': 8, 'I': 9, 'L': 10, 'K': 11, 'M': 12, 'F': 13, 'P': 14, 'S': 15, 'T': 16, 'W': 17, 'Y': 18, 'V': 19, 'X': 20, '-': 20}
IDs,seqs = load_fasta(os.path.join(MSA_saved_dir, "msa.a3m"))
seqs =[remove_insertions(seq).upper() for seq in seqs]
msa = np.array([[aa_order[aa] for aa in seq] for seq in seqs])
msa_df = mk_msa_df(msa)
job_base_dir = f"{base_output_dir}{jobname}"
n_iters=2
for iter in np.arange(1,1+n_iters):
    if iter == 1:
        saved_dir = os.path.join(job_base_dir, "bss_res")
        save_dir = os.path.join(job_base_dir, "enhanced_iter1_res")
    else:
        saved_dir = os.path.join(job_base_dir, f"enhanced_iter{iter-1}_res")
        save_dir = os.path.join(job_base_dir, f"enhanced_iter{iter}_res")
    tf.reset_default_graph()
    mrf_one = GREMLIN(msa_df,opt_iter=100)
    if N > 300:
        samp_sizes = [20,100]
    else:
        samp_sizes = [20]
    pdb_files = {}
    save_msa_ss_dirs = {}
    run_coevol = True
    for model in modle_list:
        save_msa_ss_dir = save_dir+"/msa_ss/model_%d"%model
        save_msa_fig_dir = save_dir+"/res_fig/model_%d"%model
        os.makedirs(save_msa_ss_dir, exist_ok=True)
        os.makedirs(save_msa_fig_dir, exist_ok=True)
        msa_end = "_relaxed_"
        if iter ==1:
            pattern = f"**/*_relaxed*model_{model}*.pdb"
        else:
            pattern = f"**/**/*_relaxed*model_{model}*.pdb"
        pdb_files_model =glob.glob(os.path.join(saved_dir, pattern))
        save_msa_ss_dirs_model= [re.sub("_colab","",re.sub("/pdb", "/msa" ,file[:file.index(msa_end)]+".a3m")) for file in pdb_files_model]
        save_msa_ss_dirs["model%d"%model] = save_msa_ss_dirs_model
        pdb_files["model%d"%model] = pdb_files_model
        ### extract the cluster centers of the pdbs by contact map #####
        contacts = [get_contacts(pdb_file) for pdb_file in pdb_files_model]
        mdl = PCA(n_components=0.9, random_state=42)
        embedding = mdl.fit_transform(np.array(contacts))
        n_coreset = 5
        _,msa_dirs_indx = coreset_sampling2(embedding,n_coreset)
        msa_dirs_selected = [save_msa_ss_dirs_model[idx] for idx in msa_dirs_indx]
        MRF_lliks_coevol = []
        for msa_dir in msa_dirs_selected:
            IDs_sub,seqs_sub = load_fasta(msa_dir)
            seqs_sub =[remove_insertions(seq).upper() for seq in seqs_sub]
            msa_sub = np.array([[aa_order[aa] for aa in seq] for seq in seqs_sub])
            msa_df = mk_msa_df(msa_sub)
            tf.reset_default_graph()
            mrf_coevol = GREMLIN(msa_df,opt_iter=100)
            MRF_lliks_coevol.append(GREMLIN_llik(msa, mrf_coevol))
        MRF_lliks_coevol = np.array(MRF_lliks_coevol)
        ### with coevol
        if run_coevol:
            for i in range(n_coreset):
                for j in range(n_coreset):
                    if i!=j:
                        ranked_seq = np.argsort(MRF_lliks_coevol[i,:]-MRF_lliks_coevol[j,:])
                        for s in samp_sizes:
                            msa_ss_seqs = [seqs[0]]
                            msa_ss_seqs.extend([seqs[idx] for idx in ranked_seq[0:s]])
                            IDs_ss = [IDs[0]]
                            IDs_ss.extend([IDs[idx] for idx in ranked_seq[0:s]])
                            write_fasta(IDs_ss, msa_ss_seqs, outfile=save_msa_ss_dir+'/ss_MRF_%d_2_MRF_%d_size_%03d'%(i,j,s) +'.a3m') 
        output_dir=f"{save_dir}/pdb_ss_colab/model_{model}/"
        res = ColabFold_batch(save_msa_ss_dir, output_dir,models=[model])

### Extract Representative Struc

In [32]:
## full set predictions
res = ColabFold_batch(f"{MSA_saved_basedir}{jobname}/", f"{MSA_saved_basedir}{jobname}/pdb/", models=model_list)
outputs_full = []
for model in model_list:
    o = {}
    pattern = f"*_relaxed*model_{model:01d}*.pdb"
    pdb_files = glob.glob(os.path.join(f"{MSA_saved_dir}/pdb", pattern))
    score_file_pattern = f"*_model_{model:01d}*.json"
    score_file = glob.glob(os.path.join(f"{MSA_saved_dir}/pdb", score_file_pattern))[0]
    with open(score_file,"r") as f:
        plddt_scores = pd.read_json(f)
    avg_plddt = np.mean(plddt_scores["plddt"])/100
    o.update({'msa_path': f"{MSA_saved_dir}msa.a3m"})
    o.update({'pdb_path': pdb_files[0]})
    o.update({'score_path': score_file})
    o.update({'model': model})
    o.update({'avg_plddt': avg_plddt})
    o.update({'avg_pae': np.mean(np.mean(np.mean(np.array(plddt_scores["pae"]))))})
    o.update({'max_pae': plddt_scores["max_pae"].iloc[0]})
    o.update({'ptm': plddt_scores["ptm"].iloc[0]})
    o.update
    outputs_full.append(o)
outputs_full = pd.DataFrame.from_records(outputs_full)


In [16]:
## pool preds
pool_BSS_preds(jobname,base_output_dir,lambs = lamb_list, model_list=model_list)
pool_enhanced_preds(jobname,base_output_dir,n_iters,model_list,n_coreset = 5)

In [25]:
### identify the variable region
fsr_identify_res = fsr_identify(jobname,base_output_dir)
fsr_len = fsr_identify_res['fsr_pred_resi'][1] - fsr_identify_res['fsr_pred_resi'][0]
start_res,end_res = extend_interval_symmetric(fsr_identify_res['fsr_pred_resi'][0], fsr_identify_res['fsr_pred_resi'][1], 1, L)

In [49]:
outputs_SMICE = pd.read_json(f"{base_output_dir}{jobname}/outputs_SMICE.json.zip")
filtered_data = outputs_SMICE[outputs_SMICE['avg_plddt']>0.5]

# PCA_visualization: visualized the selected representative structures with PCA
extract_rep_strucs(jobname,filtered_data,outputs_full,base_output_dir,start_res, end_res,PCA_visualization=True)

Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
Removing temporary files
